In [1]:
from src.utils import get_data_env
from src.dataloading import EpiConfig, EpiDataOrchestrator
# get data
disease_name    = 'influenza'
nuts_level      = 'nuts3'
min_date        = '2011-06-01'
max_date        = '2020-06-01'
split_trainval  = '2018-06-01'
split_valtest   = '2019-06-01'
split_berlin    = False

horizon_size    = 1
horizon_leadtime= 4
sequence_length = 1
lag_num         = 1

config = EpiConfig(
    disease             = disease_name,
    min_date            = min_date, 
    max_date            = max_date,
    horizon_size        = horizon_size,
    sequence_length     = sequence_length,
    horizon_leadtime    = horizon_leadtime,
    lag_num             = lag_num,
    nuts_level          = nuts_level,
    log_transform       = ['incidence'],
    split_berlin        = split_berlin,
    split_trainval      = split_trainval, 
    split_valtest       = split_valtest,
    target_column       = 'incidence',
    lag_column          = 'incidence',  
    verbose             = 0,

    feature_gisd        = True,
    feature_popdens     = True
    )    

# data_orchestrator = (EpiDataOrchestrator(config).build())

epidata_orchestrator  = (EpiDataOrchestrator(config)
                      .load_raw()
                      .harmonize_raw()
                      .process_data()
                      .build_features()
                      .normalize()
                      .finalize()
                      )

In [2]:
from src.dataloading import BaseLineDataLoaderManager, GraphDataLoaderManager
baselinedata             = BaseLineDataLoaderManager(epidata_orchestrator)

n_epochs        = 500
lr              = 0.0001
min_delta       = 0.0001
loss            = 'mse'

global_hparams = {
    "lr"                : lr,
    "n_epochs"          : n_epochs,
    "scheduler"         : 'plateau',
    "scheduler_kwargs"  :  {'mode': 'min', 'factor': 0.5, 'patience': 6},
    'min_delta'         : min_delta,
    'loss'              : loss,
    'patience'          : 20,
    }

graphdataloader_ig       = GraphDataLoaderManager(epidata_orchestrator).retrieve_static_graph('identity_graph').build()
graphdataloader_gn       = GraphDataLoaderManager(epidata_orchestrator).retrieve_static_graph('geographical_neighbors1').build()
# graphdataloader_cm11     = GraphDataLoaderManager(epidata_orchestrator).retrieve_static_graph('static_commuter14_1').build()
# graphdataloader_cm12     = GraphDataLoaderManager(epidata_orchestrator).retrieve_static_graph('static_commuter14_2').build()
# graphdataloader_cm13     = GraphDataLoaderManager(epidata_orchestrator).retrieve_static_graph('static_commuter14_3').build()
# graphdataloader_cm14     = GraphDataLoaderManager(epidata_orchestrator).retrieve_static_graph('static_commuter14_4').build()
# graphdataloader_cm21     = GraphDataLoaderManager(epidata_orchestrator).retrieve_static_graph('static_commuter24_1').build()
# graphdataloader_cm22     = GraphDataLoaderManager(epidata_orchestrator).retrieve_static_graph('static_commuter24_2').build()
# graphdataloader_cm23     = GraphDataLoaderManager(epidata_orchestrator).retrieve_static_graph('static_commuter24_3').build()
# graphdataloader_cm24     = GraphDataLoaderManager(epidata_orchestrator).retrieve_static_graph('static_commuter24_4').build()

In [3]:
from src.models import SimpleGCNModel

In [4]:
from src.models import ClimaScaleModel, PersistenceModel, ClimateologyModel 
from src.evaluation import Evaluator

persistence = PersistenceModel(baselinedata)
persistence.forecast()

climateology = ClimateologyModel(baselinedata)
climateology.forecast()

climascale = ClimaScaleModel(baselinedata)
climascale.forecast()

baseline_models = [persistence, climateology, climascale]

In [ ]:
global_hparams = {
    'lr'    :   1e-4,
    'min_delta' : 0.001,
    'n_epochs': 250,
    'patience':15,
    'scheduler':'plateau',
    'scheduler_kwargs': {'mode': 'min', 'factor': 0.5, 'patience': 5},
    }



ml1 = SimpleGCNModel(graphdataloader_ig, name = 'model1',verbose = 2)
ml1.set_model_hparams()
ml1.set_global_hparams(**global_hparams)
ml1.train()


==                    model1                    ==
model_hparams_set ✓
global_hparams_set ✓
Dataloader Snapshot: GraphData(x=(400, 5, 1), y=(400, 1), edge_index=(2, 400), edge_weight=(400,))
Epoch 001 train loss: 0.9893, val loss: 2.0502 ✓ (new best)
Epoch 002 train loss: 0.9255, val loss: 1.8283 ✓ (new best)
Epoch 003 train loss: 0.8408, val loss: 1.5474 ✓ (new best)
Epoch 004 train loss: 0.7613, val loss: 1.3457 ✓ (new best)
Epoch 005 train loss: 0.7173, val loss: 1.2541 ✓ (new best)
Epoch 006 train loss: 0.6950, val loss: 1.2129 ✓ (new best)
Epoch 007 train loss: 0.6732, val loss: 1.1836 ✓ (new best)
Epoch 008 train loss: 0.6641, val loss: 1.1649 ✓ (new best)
Epoch 009 train loss: 0.6537, val loss: 1.1493 ✓ (new best)
Epoch 010 train loss: 0.6481, val loss: 1.1374 ✓ (new best)
Epoch 011 train loss: 0.6412, val loss: 1.1249 ✓ (new best)
Epoch 012 train loss: 0.6378, val loss: 1.1130 ✓ (new best)
Epoch 013 train loss: 0.6282, val loss: 1.0977 ✓ (new best)
Epoch 014 train loss: 0.6234

In [ ]:
ml2 = SimpleGCNModel(graphdataloader_gn,name = 'model2',verbose = 2)
ml2.set_model_hparams()
ml2.set_global_hparams(**global_hparams)
ml2.train()